# 1. Simulated data set: 10,000 families (A; D1, D2)

Each family is generated exactly as for the MPE manuscript (`tr_aligner.simulate.TCBBSimulator`):

* ancestor A of about 50s copies of a random 11-mer, s = 2 to 20, carrying 15 motif variants;
* two descendants D1 and D2, each 5t events from A, t = 1 to 20, at deletion : substitution : duplication = 1 : 5 : 20, duplications of 1 to 5 copies and deletions of 1 to 3 copies;
* (s, t) drawn uniformly from the 19 × 20 grid for every family.

The families are split 80 / 10 / 10 into training, validation and test sets, stratified by event density (low if 5t < √(2 n(A) ln 2)). The 380 families with k = 2 of the MPE manuscript are regenerated as a second, independent test set (same seed, hence the same sequences as in the manuscript).

Runtime: CPU only, a few minutes.

In [ ]:
#@title Setup: fetch the code and choose where results are kept
REPO_URL = "https://github.com/AtheerMatroud/TandemRepeats"  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}

import os, sys, subprocess, zipfile, io
if not os.path.exists("/content/TandemRepeats/AI/tr_ai"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/TandemRepeats"])
if not os.path.exists("/content/TandemRepeats/AI/tr_ai"):
    from google.colab import files
    print("The AI folder was not found in the repository. Upload AI.zip instead.")
    up = files.upload(); name = next(iter(up))
    os.makedirs("/content/TandemRepeats", exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(up[name])) as z: z.extractall("/content/TandemRepeats")
AI = "/content/TandemRepeats/AI"
sys.path.insert(0, AI)
os.environ["PYTHONPATH"] = AI + ":" + os.environ.get("PYTHONPATH", "")
if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/tandem_repeats/ai_experiment"
else:
    ROOT = "/content/ai_experiment"
os.makedirs(ROOT, exist_ok=True)
import json, glob, time, numpy as np, pandas as pd
print("code:", AI, "| results:", ROOT, "| CPU cores:", os.cpu_count())


In [ ]:
#@title Parallel settings
# One session already uses every CPU core. To go faster, open this notebook in several
# Colab sessions at once and give each a different PART (0, 1, ..., N_PARTS - 1).
# All sessions write to the same Drive folder; finished shards are skipped, so an
# interrupted session can simply be run again.
PART = 0     #@param {type:"integer"}
N_PARTS = 1  #@param {type:"integer"}


In [ ]:
#@title Generate the 10,000 families (parallel, resumable)
N_FAMILIES = 10000  #@param {type:"integer"}
BASE_SEED = 2026    #@param {type:"integer"}
from tr_ai import simdata
t0 = time.time()
simdata.generate(ROOT, n=N_FAMILIES, base_seed=BASE_SEED, shard_size=100, part=PART, n_parts=N_PARTS)
simdata.generate_mpe_grid(ROOT)
print(f"done in {(time.time() - t0) / 60:.1f} min")


In [ ]:
#@title Split 80 / 10 / 10 (stratified by event density) and summarise
fams = simdata.load(f"{ROOT}/families/shard_*.jsonl.gz")
assert len(fams) == N_FAMILIES, f"{len(fams)} of {N_FAMILIES} families present: let every PART finish first"
splits = simdata.make_splits(fams)
json.dump(splits, open(f"{ROOT}/splits.json", "w"))
where = {i: k for k, v in splits.items() for i in v}
df = pd.DataFrame([{"split": where[f["id"]], "low": f["low"], "s": f["s"], "t": f["t"], "n(A)": len(f["A"]),
                    "n(D1)": len(f["D1"]), "n(D2)": len(f["D2"]), "variants": len(f["variants"]),
                    "A-only variants": len(f["variants"]) - f["nD"], "d_hist": np.mean(f["d_hist"])} for f in fams])
display(df.groupby(["split", "low"]).agg(families=("s", "size"), mean_nA=("n(A)", "mean"), mean_nD=("n(D1)", "mean"),
                                         mean_variants=("variants", "mean"), mean_d_hist=("d_hist", "mean")).round(1))
print("families whose ancestor has a variant absent from both descendants:", int((df["A-only variants"] > 0).sum()))


In [ ]:
#@title Check: the regenerated MPE families are those of the manuscript
from tr_ai import evaluate as ev
mpe = simdata.load(f"{ROOT}/mpe_grid/s*.jsonl.gz")
rec = ev.mpe_frame(f"{AI}/data/mpe_k2_results.csv").set_index("id")
bad = [f["id"] for f in mpe if len(f["A"]) != rec.loc[f["id"], "len_A"] or abs(f["d_hist"][0] - rec.loc[f["id"], "d_hist_0"]) > 1e-9]
print(len(mpe), "MPE families regenerated;", len(bad), "differ from the recorded results")
assert not bad
